Projeto: Merca Data Platform

Squad 2 | Funções Utilitárias Centralizadas
> Este notebook centraliza todas as funções reutilizáveis do projeto. Deve ser chamado via %run pelos demais notebooks.

In [0]:
import os
import io
import time
import logging
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv
from azure.identity import ClientSecretCredential
from azure.storage.filedatalake import DataLakeServiceClient

# ─────────────────────────────────────────────
# CONFIGURAÇÃO DE LOGS
# ─────────────────────────────────────────────
logging.basicConfig(
    level  = logging.INFO,
    format = "%(asctime)s [%(levelname)s] %(message)s"
)
log = logging.getLogger("squad2")

# ─────────────────────────────────────────────
# CARREGAMENTO DE CREDENCIAIS
# ─────────────────────────────────────────────
load_dotenv()

# ADLS
ADLS_CLIENT_ID       = os.getenv("ADLS_CLIENT_ID")
ADLS_TENANT_ID       = os.getenv("ADLS_TENANT_ID")
ADLS_CLIENT_SECRET   = os.getenv("ADLS_CLIENT_SECRET")
ADLS_STORAGE_ACCOUNT = os.getenv("ADLS_STORAGE_ACCOUNT")
ADLS_CONTAINER       = os.getenv("ADLS_CONTAINER")

# Container do Squad 2 para escrita
SQUAD2_CONTAINER = os.getenv("SQUAD2_CONTAINER", "squad2")

# SQL Server
SQL_HOST     = os.getenv("SQL_HOST")
SQL_DATABASE = os.getenv("SQL_DATABASE")
SQL_USERNAME = os.getenv("SQL_USERNAME")
SQL_PASSWORD = os.getenv("SQL_PASSWORD")

# Opções SQL reutilizáveis
SQL_OPTIONS = {
    "host"    : SQL_HOST,
    "database": SQL_DATABASE,
    "user"    : SQL_USERNAME,
    "password": SQL_PASSWORD
}

# ─────────────────────────────────────────────
# CONFIGURAÇÕES DO PROJETO
# ─────────────────────────────────────────────

# Caminhos Medalhão — VERIFICAR AO DECORRER DO PROJETO COMO VAMOS NOMEAR AS ETAPAS DO PIPELINE.
PATHS = {
    "raw"        : "real-time-data",
    "bronze"     : "squad2/bronze",
    "silver"     : "squad2/silver",
    "gold"       : "squad2/gold",
    "checkpoint" : "squad2/checkpoints"
}

# Tabelas sob responsabilidade do Squad 2 (INCLUIR AS DEMAIS TABELAS DO TIME)
TABELAS_SQUAD2 = [
    "ecommerce_categorias",
    "ecommerce_itens_pedido",
    "ecommerce_produtos"
]

# Schema destino SQL Server
SQL_SCHEMA = "squad2"
SQL_PREFIX = ""

# ─────────────────────────────────────────────
# FUNÇÕES — ADLS
# ─────────────────────────────────────────────

def get_adls_client() -> DataLakeServiceClient:
    """
    Cria e retorna um cliente autenticado do ADLS Gen2
    via Service Principal.
    """
    credential = ClientSecretCredential(
        tenant_id     = ADLS_TENANT_ID,
        client_id     = ADLS_CLIENT_ID,
        client_secret = ADLS_CLIENT_SECRET
    )
    return DataLakeServiceClient(
        account_url = f"https://{ADLS_STORAGE_ACCOUNT}.dfs.core.windows.net",
        credential  = credential
    )


def get_container_client():
    """
    Retorna o cliente do container configurado.
    """
    return get_adls_client().get_file_system_client(ADLS_CONTAINER)

def get_squad2_client():
    """
    Retorna o cliente do container squad2 para escrita
    de dados nas camadas bronze, silver e gold.
    """
    return get_adls_client().get_file_system_client(SQUAD2_CONTAINER)


def listar_snapshots(base_path: str = None) -> set:
    """
    Lista todas as pastas de snapshot disponíveis no lake.
    Padrão esperado: vendas_raw/YYYY/MM/DD/HHMMSS

    Returns:
        set: conjunto de snapshot_ids no formato YYYY/MM/DD/HHMMSS
    """
    if base_path is None:
        base_path = PATHS["raw"]

    snapshots        = set()
    container_client = get_container_client()
    paths            = container_client.get_paths(
        path      = base_path,
        recursive = True
    )

    for item in paths:
        partes = item.name.replace(base_path + "/", "").split("/")
        if len(partes) == 4 and item.is_directory:
            snapshots.add("/".join(partes))

    return snapshots


def ler_parquet(snapshot_id: str, tabela: str) -> "pyspark.sql.DataFrame":
    """
    Lê um arquivo parquet de um snapshot específico do ADLS
    diretamente na memória e retorna um Spark DataFrame.

    Args:
        snapshot_id : caminho do snapshot ex: 2026/04/27/225222
        tabela      : nome da tabela sem extensão

    Returns:
        DataFrame Spark com os dados da tabela
    """
    base_path        = PATHS["raw"]
    file_path        = f"{base_path}/{snapshot_id}/{tabela}.parquet"
    container_client = get_container_client()
    file_client      = container_client.get_file_client(file_path)

    bytes_data = file_client.download_file().readall()
    pdf        = pd.read_parquet(io.BytesIO(bytes_data))

    return spark.createDataFrame(pdf)


# ─────────────────────────────────────────────
# FUNÇÕES — SQL SERVER
# ─────────────────────────────────────────────

def get_destino_sql(tabela: str) -> str:
    """
    Retorna o nome completo da tabela destino no SQL Server.
    Formato: squad2.nome_tabela

    Args:
        tabela: nome da tabela

    Returns:
        str: nome completo schema.tabela
    """
    return f"{SQL_SCHEMA}.{SQL_PREFIX}{tabela}"


def gravar_sql(
    df     : "pyspark.sql.DataFrame",
    tabela : str,
    mode   : str = "overwrite"
) -> bool:
    """
    Grava um Spark DataFrame no SQL Server.

    Args:
        df     : DataFrame Spark a ser gravado
        tabela : nome da tabela destino
        mode   : overwrite | append (default: overwrite)

    Returns:
        bool: True se sucesso, False se erro
    """
    destino = get_destino_sql(tabela)
    try:
        df.write \
            .format("sqlserver") \
            .options(**SQL_OPTIONS) \
            .option("dbtable", destino) \
            .mode(mode) \
            .save()

        log.info(f"Gravado: {destino} → {df.count()} linhas")
        return True

    except Exception as e:
        log.error(f"Erro ao gravar {destino}: {str(e)}")
        return False


def ler_sql(tabela: str) -> "pyspark.sql.DataFrame":
    """
    Lê uma tabela do SQL Server e retorna um Spark DataFrame.

    Args:
        tabela: nome da tabela

    Returns:
        DataFrame Spark
    """
    destino = get_destino_sql(tabela)
    return spark.read \
        .format("sqlserver") \
        .options(**SQL_OPTIONS) \
        .option("dbtable", destino) \
        .load()


def validar_gravacao(tabela: str) -> bool:
    """
    Valida se uma tabela foi gravada corretamente no SQL Server.

    Args:
        tabela: nome da tabela

    Returns:
        bool: True se tabela existe e tem registros
    """
    try:
        df    = ler_sql(tabela)
        count = df.count()
        log.info(f"Validado: {get_destino_sql(tabela)} → {count} linhas")
        return count > 0
    except Exception as e:
        log.error(f"Erro ao validar {tabela}: {str(e)}")
        return False


# ─────────────────────────────────────────────
# FUNÇÕES — DELTA LAKE (Unity Catalog)
# ─────────────────────────────────────────────

def get_nome_delta(camada: str, tabela: str) -> str:
    """
    Retorna o nome completo da Delta Table no Unity Catalog.
    Formato: squad2.{camada}_{tabela}

    Args:
        camada : bronze | silver | gold
        tabela : nome da tabela

    Returns:
        str: ex: squad2.bronze_ecommerce_categorias
    """
    return f"squad2.{camada}_{tabela}"


def gravar_delta(
    df          : "pyspark.sql.DataFrame",
    camada      : str,
    tabela      : str,
    mode        : str = "append",
    particionar : bool = True
) -> bool:
    """
    Grava um Spark DataFrame como Delta Table no ADLS
    via deltalake-python.

    Args:
        df          : DataFrame Spark
        camada      : bronze | silver | gold
        tabela      : nome da tabela
        mode        : append (default) — overwrite apenas na primeira vez
        particionar : se True particiona por ano/mes/dia/hora

    Returns:
        bool: True se sucesso, False se erro
    """
    import pyarrow as pa
    from deltalake.writer import write_deltalake

    path         = get_delta_path(camada, tabela)
    storage_opts = get_storage_options()
    modo_real    = mode if delta_existe(camada, tabela) else "overwrite"

    try:
        pdf          = df.toPandas()
        tabela_arrow = pa.Table.from_pandas(pdf)

        # Particionamento apenas na Bronze
        partition_by = None
        if particionar and camada == "bronze":
            partition_by = [
                "ingestion_year",
                "ingestion_month",
                "ingestion_day",
                "ingestion_hour"
            ]
            # Verifica se colunas existem
            colunas = pdf.columns.tolist()
            if not all(c in colunas for c in partition_by):
                partition_by = None

        write_deltalake(
            table_or_uri    = path,
            data            = tabela_arrow,
            mode            = modo_real,
            storage_options = storage_opts,
            partition_by    = partition_by
        )

        log.info(
            f"Gravado: {path} → {len(pdf)} linhas | "
            f"modo: {modo_real} | "
            f"partições: {partition_by}"
        )
        return True

    except Exception as e:
        log.error(f"Erro ao gravar {path}: {str(e)}")
        return False


def ler_delta(camada: str, tabela: str) -> "pyspark.sql.DataFrame":
    """
    Lê uma Delta Table do Unity Catalog.

    Args:
        camada : bronze | silver | gold
        tabela : nome da tabela

    Returns:
        DataFrame Spark
    """
    nome_delta = get_nome_delta(camada, tabela)
    return spark.table(nome_delta)


def get_checkpoint(camada: str, tabela: str) -> tuple:
    """
    Retorna o cliente e path do checkpoint de uma tabela.

    Args:
        camada : bronze | silver | gold
        tabela : nome da tabela

    Returns:
        tuple: (container_client, checkpoint_file_path)
    """
    container_client = get_container_client()
    checkpoint_file  = f"squad2/checkpoints/{camada}/{tabela}/processed_snapshots.txt"
    return container_client, checkpoint_file


def ler_checkpoint(camada: str, tabela: str) -> set:
    """
    Lê o checkpoint de snapshots já processados.

    Returns:
        set: conjunto de snapshot_ids já processados
    """
    container_client, checkpoint_file = get_checkpoint(camada, tabela)
    processados                       = set()

    try:
        file_client = container_client.get_file_client(checkpoint_file)
        download    = file_client.download_file()
        conteudo    = download.readall().decode("utf-8")
        processados = set(conteudo.strip().split("\n")) if conteudo.strip() else set()
        log.info(f"{len(processados)} snapshot(s) já processado(s)")
    except Exception:
        log.info("Nenhum checkpoint encontrado")

    return processados


def salvar_checkpoint(camada: str, tabela: str, processados: set) -> None:
    """
    Salva o checkpoint de snapshots processados no ADLS.

    Args:
        camada      : bronze | silver | gold
        tabela      : nome da tabela
        processados : conjunto de snapshot_ids processados
    """
    container_client, checkpoint_file = get_checkpoint(camada, tabela)
    file_client                       = container_client.get_file_client(checkpoint_file)
    conteudo                          = "\n".join(processados).encode("utf-8")

    try:
        file_client.get_file_properties()
        file_client.upload_data(conteudo, overwrite=True)
    except Exception:
        file_client.create_file()
        file_client.upload_data(conteudo, overwrite=True)

    log.info(f"Checkpoint salvo → {len(processados)} snapshots")
    

# ─────────────────────────────────────────────
# FUNÇÕES — UTILITÁRIAS
# ─────────────────────────────────────────────

def get_snapshot_mais_recente(base_path: str = None) -> str:
    """
    Retorna o snapshot mais recente disponível no lake.

    Returns:
        str: snapshot_id mais recente ex: 2026/04/27/231400
    """
    snapshots = listar_snapshots(base_path)
    if not snapshots:
        raise ValueError("Nenhum snapshot encontrado no lake.")
    return sorted(snapshots)[-1]


def log_inicio(notebook: str) -> datetime:
    """Loga o início da execução de um notebook."""
    inicio = datetime.now()
    log.info(f"{'='*50}")
    log.info(f"INÍCIO: {notebook}")
    log.info(f"Data  : {inicio.strftime('%Y-%m-%d %H:%M:%S')}")
    log.info(f"{'='*50}")
    return inicio


def log_fim(notebook: str, inicio: datetime) -> None:
    """Loga o fim da execução e o tempo total."""
    fim      = datetime.now()
    duracao  = (fim - inicio).seconds
    log.info(f"{'='*50}")
    log.info(f"FIM   : {notebook}")
    log.info(f"Tempo : {duracao}s")
    log.info(f"{'='*50}")


# ─────────────────────────────────────────────
# FUNÇÕES — DELTA LAKE (ADLS via deltalake-python)
# ─────────────────────────────────────────────

def get_delta_path(camada: str, tabela: str) -> str:
    """
    Retorna o path completo da Delta Table no ADLS.
    Formato: abfss://squad2@storage.dfs.core.windows.net/camada/tabela
    """
    return (
        f"abfss://{SQUAD2_CONTAINER}@{ADLS_STORAGE_ACCOUNT}"
        f".dfs.core.windows.net/{camada}/{tabela}"
    )


def get_storage_options() -> dict:
    """
    Retorna as opções de autenticação para o deltalake-python.
    """
    return {
        "account_name"  : ADLS_STORAGE_ACCOUNT,
        "tenant_id"     : ADLS_TENANT_ID,
        "client_id"     : ADLS_CLIENT_ID,
        "client_secret" : ADLS_CLIENT_SECRET
    }


def delta_existe(camada: str, tabela: str) -> bool:
    """
    Verifica se uma Delta Table já existe no ADLS.
    """
    try:
        from deltalake import DeltaTable
        DeltaTable(
            get_delta_path(camada, tabela),
            storage_options=get_storage_options()
        )
        return True
    except Exception:
        return False


def gravar_delta(
    df     : "pyspark.sql.DataFrame",
    camada : str,
    tabela : str,
    mode   : str = "append"
) -> bool:
    """
    Grava um Spark DataFrame como Delta Table no ADLS
    via deltalake-python.

    Args:
        df     : DataFrame Spark
        camada : bronze | silver | gold
        tabela : nome da tabela
        mode   : append (default) — overwrite apenas na primeira vez

    Returns:
        bool: True se sucesso, False se erro
    """
    import pyarrow as pa
    from deltalake.writer import write_deltalake

    path         = get_delta_path(camada, tabela)
    storage_opts = get_storage_options()
    modo_real    = mode if delta_existe(camada, tabela) else "overwrite"

    try:
        # Converte Spark → pandas → PyArrow
        pdf          = df.toPandas()
        tabela_arrow = pa.Table.from_pandas(pdf)

        write_deltalake(
            table_or_uri    = path,
            data            = tabela_arrow,
            mode            = modo_real,
            storage_options = storage_opts
        )

        log.info(f"Gravado: {path} → {len(pdf)} linhas | modo: {modo_real}")
        return True

    except Exception as e:
        log.error(f"Erro ao gravar {path}: {str(e)}")
        return False


def ler_delta(camada: str, tabela: str) -> "pyspark.sql.DataFrame":
    """
    Lê uma Delta Table do ADLS via deltalake-python
    e retorna um Spark DataFrame.
    """
    import pandas as pd
    from deltalake import DeltaTable

    path         = get_delta_path(camada, tabela)
    storage_opts = get_storage_options()

    dt  = DeltaTable(path, storage_options=storage_opts)
    pdf = dt.to_pandas()

    return spark.createDataFrame(pdf)


def get_nome_delta(camada: str, tabela: str) -> str:
    """
    Retorna o path completo da Delta Table.
    Mantido por compatibilidade.
    """
    return get_delta_path(camada, tabela)

    
# ─────────────────────────────────────────────
# VALIDAÇÃO DO CARREGAMENTO
# ─────────────────────────────────────────────
def _validar_credenciais() -> None:
    credenciais = {
        "ADLS_CLIENT_ID"      : ADLS_CLIENT_ID,
        "ADLS_TENANT_ID"      : ADLS_TENANT_ID,
        "ADLS_CLIENT_SECRET"  : ADLS_CLIENT_SECRET,
        "ADLS_STORAGE_ACCOUNT": ADLS_STORAGE_ACCOUNT,
        "ADLS_CONTAINER"      : ADLS_CONTAINER,
        "SQL_HOST"            : SQL_HOST,
        "SQL_DATABASE"        : SQL_DATABASE,
        "SQL_USERNAME"        : SQL_USERNAME,
        "SQL_PASSWORD"        : SQL_PASSWORD
    }
    todas_ok = True
    for nome, valor in credenciais.items():
        if not valor:
            log.error(f"Credencial não encontrada: {nome}")
            todas_ok = False

    if todas_ok:
        log.info("Helpers carregados! Todas as credenciais OK.")
    else:
        raise EnvironmentError(
            "Credenciais ausentes. Verifique o arquivo .env"
        )

_validar_credenciais()